# L1d: Building a Floating-Point Report

Wednesday's lecture pulled a `Float64` apart by hand, one `let` block at a time. Today we turn that procedure into a single callable interface that any later notebook can import.

> __Learning Objectives:__
>
> By the end of this lab, you should be able to:
> * __Turn a procedure into an interface:__ Convert the by-hand decomposition from `L1c` into one documented function whose return value states every field it extracted.
> * __Slice a bit pattern into its fields:__ Extract the sign bit, the eleven biased-exponent bits, and the fifty-two stored fraction bits from a `Float64`, and explain why the field widths are fixed.
> * __Verify a representation by round trip:__ Rebuild the original value from the extracted fields alone, and use exact equality as the test that the decomposition was correct.

Let's get started!
___

## Setup, Data, and Prerequisites

First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:

In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

Besides Julia's `Base` library, this lab uses [the `Test` standard library](https://docs.julialang.org/en/v1/stdlib/Test/). `Include.jl` also pulls in [`src/Compute.jl`](src/Compute.jl), which is where the function you are about to write lives.

___

## What the function has to do

[The `float64_report(...)` function](src/Compute.jl) takes one `Float64` and returns a named tuple describing its components. Three of the fields are slices of the bit pattern, one is the local spacing, and one is the value rebuilt from the other three.

> __The contract:__
>
> * `value` is the argument, returned unchanged so the report is self-describing.
> * `sign_bit` is a single `Char`, position 1 of the bit pattern.
> * `exponent_bits` is an 11-character `String`, positions 2 through 12.
> * `fraction_bits` is a 52-character `String`, positions 13 through 64.
> * `spacing` is the distance to the next representable neighbour, from [the `eps(...)` function](https://docs.julialang.org/en/v1/base/base/#Base.eps-Tuple%7BAbstractFloat%7D).
> * `reconstructed` is the value rebuilt from the sign, exponent, and fraction alone.

That last field is the interesting one. If the decomposition is right, the rebuilt number must equal the original __exactly__, not approximately. 

This is the one place in the course where exact floating-point equality is the correct test, because we are checking a lossless round trip rather than the result of arithmetic.
___

## Implement the function

The function lives in [`src/Compute.jl`](src/Compute.jl), which currently holds a signature, a docstring, and four `TODO` comments.

> __The algorithm, which is Wednesday's derivation:__
>
> 1. Get the 64-character pattern with [the `bitstring(...)` function](https://docs.julialang.org/en/v1/base/numbers/#Base.bitstring).
> 2. Slice out the three fields at positions 1, 2:12, and 13:64.
> 3. Rebuild the value as $x = S\times\text{significand}\times 2^{E-1023}$, where $S = (-1)^{d_{63}}$, $E$ is the exponent bits read as a base-2 integer, and the significand is $1$ plus the weighted fraction bits.
> 4. Return the named tuple, using `eps(value)` for the spacing.

Two Julia details you will need:

> __Reading bits as numbers:__
>
> * [The `parse(...)` function](https://docs.julialang.org/en/v1/base/numbers/#Base.parse) converts a string to a number, and its `base` keyword reads a string of binary digits directly: `parse(UInt64, exponent_bits; base = 2)`.
> * Indexing a `String` with a range gives a `String`, but indexing with a single position gives a `Char`. `fraction_bits[k]` is therefore a character, and needs parsing before it can be multiplied.

Open the file, complete all four `TODO`s, then restart the kernel and run this notebook from the top. The test cell at the end is your specification. Until then the next cell stops with a "not implemented yet" error, which is the expected starting state.

Let's report on a number whose bits we already studied in `L1c`:

In [ ]:
report = float64_report(-65.78912)

The three field widths should read 1, 11, and 52: that is the whole of a `Float64`, with nothing left over. Let's confirm the arithmetic, and confirm the round trip:

In [ ]:
(sign_width = length(string(report.sign_bit)),
 exponent_width = length(report.exponent_bits),
 fraction_width = length(report.fraction_bits),
 total_bits = 1 + length(report.exponent_bits) + length(report.fraction_bits),
 round_trip_is_exact = report.reconstructed == report.value)

Sixty-four bits, and a round trip that is exact rather than close. That exactness is the evidence the decomposition is right: any error in the field boundaries, the exponent bias, or the implicit leading digit would show up here as a mismatch.

___

## The spacing field, and why equality usually fails

The `spacing` field is the one that explains the rest of the course's caution about floating point. It reports the gap between `value` and the next number a `Float64` can represent, so it is the resolution of the number line near that value. That resolution is not constant.

Let's look at how it changes with magnitude:

In [ ]:
let
    samples = [0.1, 1.0, 100.0, 1.0e6, 1.0e12]
    [(value = v, spacing = float64_report(v).spacing) for v ∈ samples]
end

The spacing grows with the magnitude of the number. Near $1$ it is about $2\times10^{-16}$; near $10^{12}$ it is about $10^{-4}$. A `Float64` does not carry a fixed number of decimal places; it carries a fixed number of significant bits, which means large numbers are stored more coarsely than small ones.

This is why `0.1 + 0.2 == 0.3` is `false`: the exact sum falls between two representable values, and the nearest one is not the one that prints as `0.3`. The round trip above was exact because nothing was computed; we took a number apart and put it back. Arithmetic is where the loss happens.

___

## Test the contract

These tests are the specification: the field widths, the exact round trip on several values including a negative one, the sign bit, and the spacing. Do they all pass?

In [ ]:
let
    @testset verbose = true "CHEME 4/5800 L1d Test Suite" begin

        @testset "field widths" begin
            r = float64_report(1.0)
            @test length(string(r.sign_bit)) == 1
            @test length(r.exponent_bits) == 11
            @test length(r.fraction_bits) == 52
        end

        @testset "sign bit" begin
            @test float64_report(1.0).sign_bit == '0'
            @test float64_report(-1.0).sign_bit == '1'
            @test float64_report(-0.1).sign_bit == '1'
        end

        @testset "exact round trip" begin
            for value ∈ (1.0, -1.0, 0.1, -0.1, 3.1415926535897, -65.78912, 1234.5)
                @test float64_report(value).reconstructed == value
            end
        end

        @testset "spacing" begin
            @test float64_report(1.0).spacing == eps(1.0)
            @test float64_report(1.0e12).spacing > float64_report(1.0).spacing
            @test 0.1 + 0.2 != 0.3
            @test isapprox(0.1 + 0.2, 0.3)
        end
    end
end;

___

## Summary
A floating-point value is three fields in a fixed-width word, and turning Wednesday's by-hand decomposition into a function makes that structure something you can query rather than something you remember.

> __Key Takeaways:__
>
> * **A procedure becomes reusable when it becomes an interface:** The same slicing and reassembly we did by hand in the lecture is worth writing once as a documented function that later notebooks can import.
> * **An exact round trip proves the decomposition:** Rebuilding the original value from the extracted fields must produce exact equality, and that is the rare case where exact floating-point comparison is the right test.
> * **Spacing is not constant:** The gap to the next representable value grows with magnitude, which is why a fixed number of significant bits does not mean a fixed number of decimal places, and why computed results need tolerant comparison.

The spacing you reported here is the quantity every `isapprox` call is implicitly reasoning about for the rest of the course.
___